# RAG and Agent Evaluation

So far, we evaluated retrieval. We checked whether search returns the
document that should answer the question.

That is only the first step. A complete application still needs to
produce a final answer. For RAG, this means checking the generated
answer. For agents, it also means looking at the tool calls the model
made before producing the answer.

RAG evaluation checks the whole flow together.

This includes:

- search
- prompt
- LLM

If the final answer is bad, the problem can come from any of these
steps. The search might retrieve the wrong document, the prompt might
omit important context, or the LLM might ignore the context.

In this part, we'll evaluate:

- RAG answers with an LLM judge
- Agent answers and tool-call trajectories

We won't go deep into agent evaluation frameworks here. We'll use the
agent from module 01, save the final answer, and also save the tool
calls. Then we can look at whether the answer is good and whether the
trajectory looks reasonable.

![RAG evaluation checks the search, prompt, LLM, and answer, while agent evaluation also checks tool calls and the trajectory before the judge](./images/11-evaluation-intro-01-rag-agent-evaluation-imagegen.png)

## LLM as a judge

For RAG and agent evaluation, we compare the generated answer with the
original answer. The generated answer won't use the same words as the
original. It's a generative model, so the phrasing will be different
even when the meaning is the same.

This is why we use another LLM to do the comparison. We show the judge
the question, the original answer, and the generated answer. Then we ask
it to decide if they are semantically equivalent.

This approach is called LLM-as-a-judge. The evaluating LLM is the
judge. It classifies each answer as good or bad and explains its
reasoning. Asking the judge to explain why it made a decision generally
produces better classifications than asking for just the verdict.

Next, we'll start with the RAG case and generate answers for the ground
truth questions.

# Generating RAG Answers

In the first part of this module, we evaluated search quality. We
checked whether the right document appeared in the search results.

Now we evaluate the full RAG pipeline. For each generated question, we
run RAG and save the answer produced by the LLM. Later, we'll compare
this answer with the original FAQ answer.

This is the A->Q->A' setup:

- A = original answer in the FAQ
- Q = generated question from this answer
- A' = answer produced by our RAG system

If A' is close to A, the RAG system is doing a good job.

This is still offline evaluation. We can compare A and A' because our
questions came from FAQ records. For each question, we know which
original answer it came from.

## Loading the data

Create a new notebook for RAG evaluation.

Load the ground truth questions:

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [6]:
q = ground_truth[10]

In [7]:
q

{'question': 'How do I join the Office Hours or live workshop if the Zoom link isn’t shared with students?',
 'document': '489dd1c9d9'}

Load the FAQ documents and the search index:

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Create a lookup table for the original FAQ documents:

In [5]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

We'll use this lookup table to find the original answer for each
ground truth question.

In [8]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

## Running RAG

Import the usual things first:

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

For this lesson, use `RAGWithUsage` from the evaluation utilities. It
subclasses `RAGBase` from module 01, so it has the same `rag` method.

It stores token usage after each LLM call. Then we can calculate the
total cost later.

It also uses the search boosts we selected in the search tuning lesson:
`question=1.0`, `answer=2.0`, and `section=0.1`.

In [10]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

For each question, `RAGBase` searches the FAQ, builds a prompt with the
retrieved context, and asks the LLM to answer. We save the answer so the
next lesson can judge it.

Run RAG for one question:

In [13]:
# rec = ground_truth[0]
# continue wiht the same question q
question = q["question"]

answer_llm = assistant.rag(question)
answer_llm

'The Zoom link is only shared with instructors/presenters/TAs, not students.\n\nAs a student, you can join the Office Hours or live workshop via:\n- YouTube Live on the DataTalksClub YouTube channel\n- The video URL posted in the announcements channel on Telegram and Slack before it starts\n- Slido for questions, with the link pinned in the chat during the live session\n\nDon’t post questions in chat, since they may be missed.'

Check the cost of this call:

In [14]:
assistant.total_cost()

0.001566

Get the original answer from the document ID:

In [15]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [16]:
print(answer_orig)

The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).

Don’t post questions in chat as they may be missed if the room is very active.


Now save both answers in one record:

In [17]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the Office Hours or live workshop if the Zoom link isn’t shared with students?',
 'answer_llm': 'The Zoom link is only shared with instructors/presenters/TAs, not students.\n\nAs a student, you can join the Office Hours or live workshop via:\n- YouTube Live on the DataTalksClub YouTube channel\n- The video URL posted in the announcements channel on Telegram and Slack before it starts\n- Slido for questions, with the link pinned in the chat during the live session\n\nDon’t post questions in chat, since they may be missed.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questi

## Processing all questions

Create a function that processes one ground truth record:

In [18]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

Test it on one record:

In [25]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course — is it still okay to join now, or did I miss the start?',
 'answer_llm': 'Yes — you can still join now. You didn’t miss the start; the course materials are available and you can begin whenever you want. If you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [26]:
answer_record = generate_rag_answer(q)
answer_record

{'question': 'How do I join the Office Hours or live workshop if the Zoom link isn’t shared with students?',
 'answer_llm': 'The Zoom link is only shared with instructors, presenters, and TAs.\n\nIf you’re a student, join via:\n\n- **YouTube Live**: the livestream is posted in the **announcements channel on Telegram and Slack** before it starts\n- **Slido**: the question link is pinned in the chat during the live session\n\nYou can also watch on the **DataTalksClub YouTube channel**.\n\nDon’t rely on the Zoom room chat for questions, since messages may be missed.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDo

In [27]:
assistant.total_cost()

0.001629

Before running the full batch, reset the usage we collected while
testing:

In [28]:
assistant.reset_usage()

In [29]:
assistant.total_cost()

0.0

This calls the LLM once per ground truth question, so it can take some
time. Let's process the questions in parallel and track progress.

Import the parallel processing helper from the same utility file:

In [21]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

Run RAG for all ground truth questions:

I had to generate for only first 200 ground_truth documents, because I hit the usage limit.

Retry is a must here!

In [34]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth[:200], generate_rag_answer)

  0%|          | 0/200 [00:00<?, ?it/s]

In [35]:
results[:10]

[{'question': 'I just found this course — is it still okay to join now, or did I miss the start?',
  'answer_llm': 'Yes, you can still join now. You can start whenever you want, and if you want a certificate, make sure to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I start late, can I still take part in the course and work through the materials?',
  'answer_llm': 'Yes. You can start whenever you want and still work through the videos, notebooks, and GitHub materials. If you want a certificate, though, you need to submit your project while a live cohort is still accepting submissions.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed t

In [36]:
df_results = pd.DataFrame(results)
df_results[:10]

,question,answer_llm,answer_orig,document
0,I just found this course — is it still okay to...,"Yes, you can still join now. You can start whe...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,"If I start late, can I still take part in the ...",Yes. You can start whenever you want and still...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,Am I allowed to join after the course has alre...,"Yes, you can still join after the course has a...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Will I still be able to get a certificate if I...,"Yes, but only if you submit your project while...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,What’s the deadline if I want to earn the cert...,You need to submit your capstone project while...,"Yes, but if you want to receive a certificate,...",74eb249bbf
5,I signed up for LLM Zoomcamp but still didn't ...,No — you don’t need to wait for a confirmation...,You don't need it. You're accepted. You can al...,977bf7786c
6,Do I need an acceptance or confirmation email ...,You don’t need an acceptance or confirmation e...,You don't need it. You're accepted. You can al...,977bf7786c
7,"If I registered for the LLM Zoomcamp, is there...",You don’t need to wait for an approval or conf...,You don't need it. You're accepted. You can al...,977bf7786c
8,Can I submit LLM Zoomcamp homework without bei...,Yes — you can submit LLM Zoomcamp homework wit...,You don't need it. You're accepted. You can al...,977bf7786c
9,What’s the point of registering for the LLM Zo...,You don’t need a confirmation message. You’re ...,You don't need it. You're accepted. You can al...,977bf7786c


`generate_rag_answer` returns one answer record for each question.

In [37]:
assistant.total_cost()

1.077527999999999

Collect the answer records:

In [38]:
answers = []

for answer_record in results:
    answers.append(answer_record)

Save the answers:

In [39]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new-short.csv", index=False)

I generated this file on Sep 22, 2026. The run used 200 out of 765 ground truth questions.

The total cost was $1.077528 (😭) with failed attempt included (running 765 ground truth questions failed due to usage limit).

If you don't want to generate the RAG answers yourself, download the file we prepared:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

wget -O data/rag-answers-new.csv ${PREFIX}/cohorts/2026/04-evaluation/data/rag-answers-new.csv
```

In the next lesson, we'll evaluate these answers with an LLM judge.

# LLM as a Judge

In the previous lesson, we generated RAG answers for our ground truth
questions. Now we need to decide whether these answers are good enough.

For offline evaluation, we have three things:

- the original FAQ answer
- the question generated from that answer
- the answer generated by our RAG pipeline

An LLM judge is another LLM call that compares these three pieces. We
ask it whether the RAG answer recovers the same information as the
original answer.

It can also explain why an answer is bad.

For example:

- the retrieved document might be wrong
- the answer might miss the key point
- the model might say that it doesn't know

This approach is useful when exact text matching is too strict. The RAG
answer doesn't need to copy the FAQ answer word for word. It needs to
answer the question with the same key information.

This evaluates the full RAG flow in one pass:

- search: did we retrieve context that contains the answer?
- prompt: did we give the model enough context to answer?
- LLM: did the model produce a useful answer from that context?

If the judge marks an answer as bad, we still need to look at the
example. The judge tells us where to investigate. It doesn't replace
reading the failing cases.


## Loading the RAG answers

Start from the CSV we created in the previous lesson:

In [41]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new-short.csv")
answers = df_answers.to_dict(orient="records")

Each row has the generated question, the original FAQ answer, and the
answer produced by the RAG pipeline.

This is offline evaluation. We can do it because our test dataset came
from FAQ records. We know the original answer for every generated
question.

In production, we usually don't have that original answer for real user
questions. There we can still use an LLM judge. The prompt has to judge
only the question and the generated answer. In this lesson, we use the
stronger offline setup.

## A->Q->A' evaluation

We'll compare the RAG answer with the original answer from the FAQ.
This checks if the RAG pipeline is producing answers that match the
ground truth.

First, define the output format:

In [42]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

The judge returns two fields. The `score` gives us a metric we can
aggregate. The `reasoning` explains the score, which helps when we look
at bad examples.

First, write the judge instructions. This tells the judge what to
compare and how to assign the score.

In [43]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

Then define the prompt template. This is the data we pass to the judge
for each answer.

In [44]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [45]:
aqa_judge_prompt

'Question:\n{question}\n\nOriginal Answer (ground truth):\n{answer_orig}\n\nAI Answer:\n{answer_llm}'

Import the structured-output helper:

In [46]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

Take one record:

In [47]:
rec = answers[0]

In [48]:
rec

{'question': 'I just found this course — is it still okay to join now, or did I miss the start?',
 'answer_llm': 'Yes, you can still join now. You can start whenever you want, and if you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

Create the judge prompt:

In [49]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [50]:
print(prompt)

Question:
I just found this course — is it still okay to join now, or did I miss the start?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes, you can still join now. You can start whenever you want, and if you want a certificate, make sure to submit your project while submissions are still being accepted.


Call the judge:

In [51]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
) # type: ignore

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key point from the ground truth: it is okay to join now, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent.', score='good')

Check the cost:

In [53]:
calc_price(usage)

{'input_cost': 0.00023024999999999999,
 'output_cost': 0.0002385,
 'total_cost': 0.00046875}

Now put the same logic into a function:

In [54]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    ) # type: ignore

    return result, usage

Test it on the same record:

In [55]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: joining now is allowed, and certificate eligibility depends on submitting the project while submissions are still open. It adds a bit of wording about starting whenever you want, but that does not change the core answer.', score='good')

## Running the judge

Run the evaluation on all answers:

In [56]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

Use the same parallel processing helper:

In [57]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/200 [00:00<?, ?it/s]

Split the results:

In [58]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

Create a dataframe:

In [59]:
df_eval = pd.DataFrame(evaluations)

Calculate the total cost:

In [60]:
calc_total_price(usages)

0.13449075

Check the results:

In [61]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 190/200 = 95.00%


Look at the "bad" cases to understand what went wrong:

In [62]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
27,Do I need to submit homework to receive the ce...,69d122f12e,bad,The AI answer correctly says homework is not r...
33,What do I actually need to complete to receive...,9f689c185f,bad,The ground truth says the certificate requires...
40,When is the next llm-zoomcamp session starting?,bd31146b0e,bad,The ground truth says the next llm-zoomcamp se...
82,"If I miss one homework assignment, does that s...",cdc3b285e5,bad,The AI answer correctly says that missing home...
84,"Is homework required for passing the course, o...",cdc3b285e5,bad,The AI answer captures the main point that hom...


These rows are often the most useful part of the evaluation. They can
show that search retrieved the wrong document. They can also show that
the answer is too generic. Sometimes the RAG pipeline says that it
doesn't know even though the FAQ had the answer.

## Evaluating the judge

The judge can be wrong. It may rate an answer as good even though search
failed to retrieve the right document. In that case the judge is too
lenient. Make the instructions stricter and re-run the evaluation.

To evaluate the judge, you need to look at the results yourself. Sample
some good and bad cases, read the judge reasoning, and check whether you
agree with the verdict. You cannot use another judge to evaluate the
judge. This is manual work, but it is necessary.

A practical approach is to build a simple application with Streamlit.
Show each question, the original answer, the generated answer, and the
judge verdict side by side. Then mark each verdict as correct or
incorrect and use that feedback to adjust the judge instructions. This
is a lot of trial and error, but it makes the evaluation framework more
reliable.

## Saving the results

Save the judged answers:

In [63]:
df_eval.to_csv("data/rag-evaluations-new-short.csv", index=False)

I generated this file on Sep 22, 2026. The run used 200 RAG answers.

The results were:

- Good: 190
- Bad: 10

The total cost was $0.134491, about 13 cents.

If you don't want to run the judge yourself, download the file we
prepared:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget -O data/rag-evaluations-new.csv ${PREFIX}/cohorts/2026/04-evaluation/data/rag-evaluations-new.csv
```

We now have an answer-quality score for the RAG pipeline. In the next
lesson, we'll apply the same idea to an agent and also capture the tool
calls it made.